# VisClick — Phase 4.3 / D-05: few-shot data-efficiency curve

**Goal.** Quantify how YOLOv8s mAP / CPV improves as we fine-tune on `k` hand-corrected desktop screens. The curve answers the proposal's sample-efficiency research question for VisClick's chosen architecture, in the small-`k` regime that matches the expected deployment workflow (a domain expert hand-labels a handful of representative screens before a bot is rolled out, not hundreds).

**Prerequisites:**
- `05_train_source.ipynb` has been run. `<DRIVE>/weights/baseline_source/best_source_v8s.pt` exists.
- Hand-corrected zip is committed at `datasets/handcorrected_desktop_test/visclick3.yolov8.zip` (same file `scripts/run_cpv.py` uses).

**Pipeline:**
1. Mount Drive → `git pull` → install `ultralytics`, `datasets`, `opencv-python`.
2. Bootstrap source weights + unzip hand-corrected; emit a deterministic sorted list of stems.
3. For each `k ∈ {1, 2, 4, 8}`: build a tiny YOLO dataset (first `k` sorted images), head-only fine-tune from source-pretrained `best_source_v8s.pt`, save weights to `<DRIVE>/weights/few_shot/k{k}/`. Resume-aware.
4. For each `k ∈ {0, 1, 2, 4, 8}`: evaluate **CPV on ScreenSpot desktop** (third-party-labelled, n=334, primary metric) and **mAP@0.5 / mAP@0.5:0.95 on the hand-corrected set** (secondary, with a `fit_to_train` flag where applicable).
5. Plot the curve and write `reports/tables/sample_efficiency.csv` + `reports/figures/sample_efficiency_curve.png`.

**Compute reality check.** Per-k fine-tune at `k ≤ 8` and `imgsz=640` is ~5–10 min on T4. ScreenSpot CPV is ~5 min per checkpoint. Total budget ~1–2 hours, comfortable in one Colab Free session.

**Report.** Every step prints `REPORT ...` lines for `VisClick_Report_Data_Form.md` and `Final_Report_GAPS.md` D-05.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, subprocess
REPO = "https://github.com/HiranMadhu/visclick.git"
ROOT = "/content/visclick"
if not os.path.isdir(os.path.join(ROOT, ".git")):
    subprocess.run(["git", "clone", REPO, ROOT], check=True)
    print("Cloned to", ROOT)
else:
    subprocess.run(["git", "-C", ROOT, "fetch", "origin"], check=False)
    subprocess.run(["git", "-C", ROOT, "pull", "--rebase", "origin", "main"], check=False)
    print("Pulled latest in", ROOT)
print("REPORT git_head =", subprocess.check_output(["git", "-C", ROOT, "rev-parse", "--short", "HEAD"], text=True).strip())

In [ ]:
import sys, subprocess
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "ultralytics", "datasets", "pillow", "opencv-python", "matplotlib"],
    check=False,
)
import torch, ultralytics
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "| device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
print("ultralytics:", ultralytics.__version__)
print("REPORT env | torch =", torch.__version__,
      "| cuda =", torch.cuda.is_available(),
      "| ultralytics =", ultralytics.__version__)

## 8.0 — Bootstrap source weights and hand-corrected set

- Source checkpoint: `<DRIVE>/weights/baseline_source/best_source_v8s.pt` (from `05_train_source.ipynb`).
- Hand-corrected: unzip `datasets/handcorrected_desktop_test/visclick3.yolov8.zip` into `/content/hc/`.
- Stems are sorted alphabetically so a `k=4` run is a strict subset of the `k=8` run → the curve is monotone-by-construction.

In [ ]:
import os, zipfile, shutil

DRIVE         = "/content/drive/MyDrive/visclick"
SOURCE_WTS    = os.path.join(DRIVE, "weights", "baseline_source", "best_source_v8s.pt")
FEWSHOT_DIR   = os.path.join(DRIVE, "weights", "few_shot")
REPORTS_TBL   = os.path.join(DRIVE, "reports", "tables")
REPORTS_FIG   = os.path.join(DRIVE, "reports", "figures")
os.makedirs(FEWSHOT_DIR, exist_ok=True)
os.makedirs(REPORTS_TBL, exist_ok=True)
os.makedirs(REPORTS_FIG, exist_ok=True)

assert os.path.isfile(SOURCE_WTS), (
    f"Source weights not found: {SOURCE_WTS}. "
    f"Run 05_train_source.ipynb first."
)
size_mb = os.path.getsize(SOURCE_WTS) / 1024 / 1024
print(f"REPORT source_weights | path = {SOURCE_WTS} | size_mb = {size_mb:0.1f}")

HC_ZIP  = "/content/visclick/datasets/handcorrected_desktop_test/visclick3.yolov8.zip"
HC_ROOT = "/content/hc"
HC_IMG  = os.path.join(HC_ROOT, "train", "images")
HC_LBL  = os.path.join(HC_ROOT, "train", "labels")

if not os.path.isdir(HC_IMG):
    assert os.path.isfile(HC_ZIP), f"Missing zip {HC_ZIP}; pull repo first."
    os.makedirs(HC_ROOT, exist_ok=True)
    with zipfile.ZipFile(HC_ZIP) as zf:
        zf.extractall(HC_ROOT)
    print("unzipped", HC_ZIP, "->", HC_ROOT)

STEMS = sorted(
    os.path.splitext(f)[0]
    for f in os.listdir(HC_IMG)
    if f.lower().endswith((".png", ".jpg", ".jpeg"))
)
N_HC = len(STEMS)
print(f"REPORT hc_pool | n = {N_HC} | first = {STEMS[:3]} | last = {STEMS[-3:]}")
assert N_HC >= 4, f"need at least 4 hand-corrected images, got {N_HC}"

CLASSES = ["button", "text", "text_input", "icon", "menu", "checkbox"]

## 8.1 — Per-`k` fine-tune loop

For each `k`, we:
1. Pick the **first `k` sorted stems** (so smaller-`k` is a strict subset of larger-`k`).
2. Build a tiny YOLO dataset under `/content/few_shot/k{k}/{images,labels}/train/` (symlinks into the unzipped hand-corrected folder — no copy).
3. Use the same `k` images for `val` (since with `k ≤ 8` we have no room for a hold-out; we use ScreenSpot for held-out evaluation). YOLO will early-stop on train loss.
4. Head-only fine-tune from `best_source_v8s.pt` with `freeze=10` (lock backbone), `epochs=50`, `patience=15`, `imgsz=640`, `batch=min(k, 4)`. AdamW, `lr0=1e-3`.
5. Save to `<DRIVE>/weights/few_shot/k{k}/run1/`.

**Resume-aware.** If `<DRIVE>/weights/few_shot/k{k}/run1/weights/best.pt` already exists, the fine-tune is skipped and the existing checkpoint is reused.

In [ ]:
import os, time, yaml
from ultralytics import YOLO

K_VALUES   = [1, 2, 4, 8]
EPOCHS     = 50
PATIENCE   = 15
IMGSZ      = 640
LR0        = 1e-3
FREEZE     = 10
FORCE_FRESH = False  # set True to ignore existing best.pt and retrain

def _build_few_shot_dataset(k: int) -> str:
    """Symlink first-k images + labels into /content/few_shot/k{k}/ and write data.yaml.
    Returns the absolute path to the data.yaml that YOLO will consume."""
    root = f"/content/few_shot/k{k}"
    img_dir = os.path.join(root, "images", "train")
    lbl_dir = os.path.join(root, "labels", "train")
    os.makedirs(img_dir, exist_ok=True)
    os.makedirs(lbl_dir, exist_ok=True)
    picked = STEMS[:k]
    for stem in picked:
        for fn in os.listdir(HC_IMG):
            if os.path.splitext(fn)[0] == stem:
                src = os.path.join(HC_IMG, fn)
                dst = os.path.join(img_dir, fn)
                if not os.path.lexists(dst):
                    os.symlink(src, dst)
                break
        lbl_src = os.path.join(HC_LBL, stem + ".txt")
        lbl_dst = os.path.join(lbl_dir, stem + ".txt")
        if os.path.isfile(lbl_src) and not os.path.lexists(lbl_dst):
            os.symlink(lbl_src, lbl_dst)
    yml = os.path.join(root, "data.yaml")
    with open(yml, "w") as fh:
        yaml.safe_dump(
            {"path": root, "train": "images/train", "val": "images/train",
             "nc": len(CLASSES), "names": CLASSES},
            fh, sort_keys=False,
        )
    return yml

def _finetune_one(k: int) -> str:
    """Run a single head-only fine-tune at sample size k. Returns path to best.pt."""
    project = os.path.join(FEWSHOT_DIR, f"k{k}")
    name    = "run1"
    run_dir = os.path.join(project, name)
    best_pt = os.path.join(run_dir, "weights", "best.pt")
    if os.path.isfile(best_pt) and not FORCE_FRESH:
        print(f"REPORT ftune | k = {k} | status = SKIP_ALREADY_TRAINED | weights = {best_pt}")
        return best_pt
    data_yaml = _build_few_shot_dataset(k)
    batch = min(k, 4)
    print(f"REPORT ftune | k = {k} | status = START | batch = {batch} | data = {data_yaml}")
    t0 = time.time()
    model = YOLO(SOURCE_WTS)
    model.train(
        data=data_yaml,
        epochs=EPOCHS, patience=PATIENCE,
        imgsz=IMGSZ, batch=batch,
        lr0=LR0, optimizer="AdamW", freeze=FREEZE,
        project=project, name=name, exist_ok=True,
        verbose=False, plots=False,
        # Workers small to dodge Colab CPU-bound symlink races on tiny datasets.
        workers=2,
    )
    elapsed = (time.time() - t0) / 60.0
    assert os.path.isfile(best_pt), f"YOLO did not write {best_pt}"
    size_mb = os.path.getsize(best_pt) / 1024 / 1024
    print(f"REPORT ftune | k = {k} | status = DONE | minutes = {elapsed:0.1f} "
          f"| best.pt = {best_pt} ({size_mb:0.1f} MB)")
    return best_pt

FT_WEIGHTS = {}
for k in K_VALUES:
    FT_WEIGHTS[k] = _finetune_one(k)
FT_WEIGHTS[0] = SOURCE_WTS
print("\nREPORT ftune | all_done")
for k in sorted(FT_WEIGHTS):
    print(f"  k={k:>2d} -> {FT_WEIGHTS[k]}")

## 8.2 — Per-`k` evaluation on ScreenSpot (primary) + hand-corrected (secondary)

Two metrics per checkpoint:
1. **CPV on ScreenSpot desktop** (n=334, third-party-labelled): same definition as `scripts/run_cpv_screenspot.py` — hit if **any** predicted box centre falls inside the GT box. Class-agnostic. Comparable across papers using the same benchmark.
2. **mAP@0.5 / mAP@0.5:0.95 on the hand-corrected set** via Ultralytics `model.val()`. For `k > 0` this is a `fit_to_train` measurement (we trained on the same images), so it is reported but flagged in the CSV. The ScreenSpot CPV is the held-out number.

ScreenSpot data is loaded once from HuggingFace (`rootsautomation/ScreenSpot`) and reused for all `k`.

In [ ]:
import os, tempfile, time
import numpy as np
from PIL import Image
from datasets import load_dataset
from ultralytics import YOLO

SCREENSPOT_HF = "rootsautomation/ScreenSpot"
SCREENSPOT_CACHE = os.path.join(tempfile.gettempdir(), "visclick_hf_cache")
os.makedirs(SCREENSPOT_CACHE, exist_ok=True)

print(f"loading {SCREENSPOT_HF} (cache={SCREENSPOT_CACHE})...")
ss = load_dataset(SCREENSPOT_HF, cache_dir=SCREENSPOT_CACHE)
split_name = next(iter(ss.keys()))
ss_rows = ss[split_name]
ss_keep = [r for r in ss_rows
           if str(r.get("data_source", "")).lower() in ("windows", "macos")]
print(f"REPORT screenspot | total = {len(ss_rows)} | desktop_slice = {len(ss_keep)}")

def _bbox_to_pixels(bbox, w, h):
    a, b, c, d = (float(x) for x in bbox)
    return a * w, b * h, c * w, d * h

def _eval_screenspot_cpv(weights: str, conf: float = 0.25, iou: float = 0.5) -> dict:
    """Run YOLOv8 over ScreenSpot desktop rows; return overall + per-slice CPV."""
    model = YOLO(weights)
    total = hit = 0
    by_type = {}
    t0 = time.time()
    for i, r in enumerate(ss_keep):
        img = r.get("image")
        if img is None:
            continue
        if not isinstance(img, Image.Image):
            img = Image.open(img)
        rgb = np.array(img.convert("RGB"))
        h, w = rgb.shape[:2]
        try:
            x1, y1, x2, y2 = _bbox_to_pixels(r["bbox"], w, h)
        except Exception:
            continue
        x1 = max(0.0, min(x1, w - 1)); x2 = max(0.0, min(x2, w - 1))
        y1 = max(0.0, min(y1, h - 1)); y2 = max(0.0, min(y2, h - 1))
        if x2 <= x1 or y2 <= y1:
            continue
        preds = model.predict(rgb, conf=conf, iou=iou, imgsz=IMGSZ, verbose=False)
        boxes = preds[0].boxes.xyxy.cpu().numpy() if preds and preds[0].boxes is not None else np.empty((0, 4))
        centres = [((bx[0] + bx[2]) / 2.0, (bx[1] + bx[3]) / 2.0) for bx in boxes]
        h_ = 1 if any(x1 <= cx <= x2 and y1 <= cy <= y2 for cx, cy in centres) else 0
        total += 1
        hit += h_
        dtype = str(r.get("data_type", "unknown"))
        by_type.setdefault(dtype, [0, 0])
        by_type[dtype][0] += 1
        by_type[dtype][1] += h_
        if (i + 1) % 100 == 0:
            print(f"    [{i+1:4d}/{len(ss_keep)}] cpv = {hit/max(total,1)*100:5.2f}%")
    return {
        "total": total, "hit": hit,
        "cpv_pct": (hit / max(total, 1)) * 100.0,
        "cpv_text_pct": (by_type.get("text", [1, 0])[1] / max(by_type.get("text", [1, 0])[0], 1)) * 100.0,
        "cpv_icon_pct": (by_type.get("icon", [1, 0])[1] / max(by_type.get("icon", [1, 0])[0], 1)) * 100.0,
        "elapsed_min": (time.time() - t0) / 60.0,
    }

HC_FULL_YAML = "/content/hc/data_eval.yaml"
if not os.path.isfile(HC_FULL_YAML):
    import yaml
    with open(HC_FULL_YAML, "w") as fh:
        yaml.safe_dump(
            {"path": "/content/hc", "train": "train/images", "val": "train/images",
             "nc": len(CLASSES), "names": CLASSES},
            fh, sort_keys=False,
        )
    print("wrote", HC_FULL_YAML)

def _eval_hc_map(weights: str) -> dict:
    """mAP on the full hand-corrected set via Ultralytics model.val()."""
    model = YOLO(weights)
    res = model.val(data=HC_FULL_YAML, imgsz=IMGSZ, batch=4, verbose=False, plots=False, save=False)
    return {
        "hc_map_50": float(getattr(res.box, "map50", float("nan"))),
        "hc_map_50_95": float(getattr(res.box, "map", float("nan"))),
    }

EVAL_ROWS = []
for k in [0] + K_VALUES:
    weights = FT_WEIGHTS[k]
    print(f"\n=== evaluating k = {k} | weights = {os.path.basename(weights)} ===")
    ss_res = _eval_screenspot_cpv(weights)
    hc_res = _eval_hc_map(weights)
    fit_flag = "held_out" if k == 0 else "fit_to_train"
    row = {
        "k": k,
        "weights": weights,
        "hc_eval_flag": fit_flag,
        "hc_map_50": round(hc_res["hc_map_50"], 4),
        "hc_map_50_95": round(hc_res["hc_map_50_95"], 4),
        "screenspot_n": ss_res["total"],
        "screenspot_cpv_pct": round(ss_res["cpv_pct"], 2),
        "screenspot_cpv_text_pct": round(ss_res["cpv_text_pct"], 2),
        "screenspot_cpv_icon_pct": round(ss_res["cpv_icon_pct"], 2),
        "screenspot_eval_min": round(ss_res["elapsed_min"], 1),
    }
    EVAL_ROWS.append(row)
    print(f"REPORT eval | k = {k} "
          f"| hc_map_50 = {row['hc_map_50']:0.4f} ({fit_flag}) "
          f"| ss_cpv = {row['screenspot_cpv_pct']:5.2f}% "
          f"({ss_res['hit']}/{ss_res['total']})")
print("\nREPORT eval | all_done")

## 8.3 — Plot the sample-efficiency curve

Two y-axes on a single figure: ScreenSpot CPV (left, primary) and hand-corrected mAP@0.5 (right, secondary, with `k > 0` points flagged as in-sample). Saved to `<DRIVE>/reports/figures/sample_efficiency_curve.png`.

In [ ]:
import os
import matplotlib.pyplot as plt

ks    = [r["k"] for r in EVAL_ROWS]
cpv   = [r["screenspot_cpv_pct"] for r in EVAL_ROWS]
hcmap = [r["hc_map_50"] * 100.0 for r in EVAL_ROWS]

fig, ax1 = plt.subplots(figsize=(7.0, 4.2))
ax1.set_xlabel("k (labelled desktop images used for fine-tune)")
ax1.set_ylabel("ScreenSpot CPV (%)  \u2014  held-out, primary", color="tab:blue")
l1 = ax1.plot(ks, cpv, "o-", color="tab:blue", label="ScreenSpot CPV (held-out)")
ax1.tick_params(axis="y", labelcolor="tab:blue")
ax1.set_xticks(ks)
ax1.grid(True, alpha=0.3)

ax2 = ax1.twinx()
ax2.set_ylabel("Hand-corrected mAP@0.5 (%)  \u2014  in-sample for k>0", color="tab:orange")
l2 = ax2.plot(ks, hcmap, "s--", color="tab:orange", label="Hand-corrected mAP@0.5")
ax2.tick_params(axis="y", labelcolor="tab:orange")

lines = l1 + l2
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc="lower right", fontsize=8)
fig.suptitle("VisClick (YOLOv8s) sample efficiency: k vs CPV / mAP")
fig.tight_layout()

fig_path = os.path.join(REPORTS_FIG, "sample_efficiency_curve.png")
fig.savefig(fig_path, dpi=150, bbox_inches="tight")
print(f"REPORT plot | path = {fig_path}")
plt.show()

## 8.4 — Write metrics CSV for the report

Single-table summary at `<DRIVE>/reports/tables/sample_efficiency.csv`. One row per `k`; fields cover both metrics, the eval-flag, and the per-slice CPV. T-02 in the gaps tracker reads this file.

In [ ]:
import csv, os

out_csv = os.path.join(REPORTS_TBL, "sample_efficiency.csv")
fieldnames = list(EVAL_ROWS[0].keys())
with open(out_csv, "w", newline="") as fh:
    w = csv.DictWriter(fh, fieldnames=fieldnames)
    w.writeheader()
    for r in EVAL_ROWS:
        w.writerow(r)
print("REPORT step = WRITE_CSV | path =", out_csv)
print()
print("=== sample_efficiency.csv ===")
header = "  ".join(f"{c:>22s}" for c in fieldnames)
print(header)
for r in EVAL_ROWS:
    print("  ".join(f"{str(r[c]):>22s}" for c in fieldnames))